# Logistic Regression - Plecoptera: Perlidae and Trichoptera: Helicopsychidae

Binary logistic regression models for the presence/absence of the two bioindicator taxa in the Cali River. Predictors are screened by AIC over a candidate set, and the selected models are validated with **leave-one-out cross-validation (LOOCV)**.

## Data Loading

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from itertools import combinations
import matplotlib.pyplot as plt

file_path = "../../data/DB - Macroinvertebrados.xlsx"
df = pd.read_excel(file_path)
df.columns = df.columns.str.strip()
df

## Predictor Selection

The candidate predictors (`DBO5`, `Dureza`, `Caudal`) come from a 3-variable PCA. All subsets are fitted as binomial GLMs and ranked by AIC. The best subset for Perlidae turned out to be `DBO5` alone.

In [ ]:
predictors = ['DBO5', 'Dureza', 'Caudal']  # candidate predictors (from a 3-variable PCA)
response = ['Perlidae']
X = df[predictors]
y = df[response]

n_variables = X.shape[1]
print(f"Maximum number of possible models: {2**n_variables - 1}")

In [ ]:
def select_model(dataframe, predictors, response):
    """Exhaustive AIC-based subset selection for a binomial GLM."""
    best_model = None
    best_aic = np.inf
    best_combination = None
    total_combinations = 0

    for L in range(1, len(predictors) + 1):
        for subset in combinations(predictors, L):
            total_combinations += 1
            X_subset = sm.add_constant(dataframe[list(subset)])
            result = sm.GLM(dataframe[response], X_subset,
                            family=sm.families.Binomial()).fit()
            print(f'Trying model with predictors: {subset}, AIC: {result.aic}')
            if result.aic < best_aic:
                best_aic = result.aic
                best_model = result
                best_combination = subset

    print(f'Total combinations tried: {total_combinations}')
    print(f'Best model has predictors: {best_combination} with AIC: {best_aic}')
    return best_model

best_model = select_model(df, predictors, response)
print(best_model.summary())

## Model Definition & Evaluation

### Perlidae - model with DBO5 (LOOCV)

Predictors are standardised; the logistic model uses an L2 penalty with balanced class weights. Both validation (held-out) and calibration (training) metrics are reported.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import confusion_matrix, classification_report
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

X = df[['DBO5']]
y = df['Perlidae']
X_scaled = StandardScaler().fit_transform(X)

model = LogisticRegression(penalty='l2', solver='liblinear', max_iter=1000,
                           random_state=42, class_weight='balanced')
loo = LeaveOneOut()

y_true_val, y_pred_val = [], []
y_true_cal, y_pred_cal = [], []

for train_index, test_index in loo.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)

    # Validation (held-out observation)
    y_true_val.append(y_test.iloc[0])
    y_pred_val.append(model.predict(X_test)[0])

    # Calibration (training observations)
    y_true_cal.extend(y_train.values.tolist())
    y_pred_cal.extend(model.predict(X_train).tolist())

print("Confusion matrix - validation (LOOCV):")
print(confusion_matrix(y_true_val, y_pred_val))
print("\nClassification report - validation (LOOCV):")
print(classification_report(y_true_val, y_pred_val))

print("Confusion matrix - calibration:")
print(confusion_matrix(y_true_cal, y_pred_cal))
print("\nClassification report - calibration:")
print(classification_report(y_true_cal, y_pred_cal))

### Trichoptera: Helicopsychidae - model with DBO5 (LOOCV)

In [ ]:
X = df[['DBO5']]
y = df['Trichoptera']
X_scaled = StandardScaler().fit_transform(X)

model_trich = LogisticRegression(penalty='l2', solver='liblinear', max_iter=1000,
                                 random_state=42, class_weight='balanced')
loo = LeaveOneOut()

y_true_val, y_pred_val = [], []
y_true_cal, y_pred_cal = [], []

for train_index, test_index in loo.split(X_scaled):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model_trich.fit(X_train, y_train)

    y_true_val.append(y_test.iloc[0])
    y_pred_val.append(model_trich.predict(X_test)[0])

    y_true_cal.extend(y_train.values.tolist())
    y_pred_cal.extend(model_trich.predict(X_train).tolist())

print("Confusion matrix - validation (LOOCV):")
print(confusion_matrix(y_true_val, y_pred_val))
print("\nClassification report - validation (LOOCV):")
print(classification_report(y_true_val, y_pred_val))

print("Confusion matrix - calibration:")
print(confusion_matrix(y_true_cal, y_pred_cal))
print("\nClassification report - calibration:")
print(classification_report(y_true_cal, y_pred_cal))

## Visualisation

Fitted logistic curves (predicted probability vs BOD5) for each taxon. To draw the curves we refit each single-predictor model on the standardised range; here the curves use the unstandardised coefficients stored after the LOOCV loop.

In [ ]:
# Helicopsychidae fitted curve
intercept = model_trich.intercept_[0]
coefficient = model_trich.coef_[0][0]
X_continuous = np.linspace(-5, df['DBO5'].max(), 300)
y_curve = 1 / (1 + np.exp(-(intercept + coefficient * X_continuous)))

plt.figure(figsize=(8, 6))
plt.scatter(df['DBO5'], df['Trichoptera'], color='blue', alpha=0.6, label='Observed data')
plt.plot(X_continuous, y_curve, color='red', linewidth=2, label='Fitted logistic curve')
plt.title('Logistic Regression Model - Trichoptera: Helicopsychidae', fontsize=14)
plt.xlabel('BOD₅ (mg/l)', fontsize=12)
plt.ylabel('Predicted probability', fontsize=12)
plt.legend(fontsize=12)
equation = f'$P(BOD_5) = \\frac{{1}}{{1 + e^{{-({intercept:.4f} + {coefficient:.4f} \\cdot BOD_5)}}}}$'
plt.text(1, 0.8, equation, fontsize=18, color='black')
plt.axvline(x=0, color='black', linestyle='--')
plt.grid(True)
plt.savefig("../../figures/logistic_Helicopsychidae.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Perlidae fitted curve
intercept = model.intercept_[0]
coefficient = model.coef_[0][0]
X_continuous = np.linspace(-5, df['DBO5'].max(), 300)
y_curve = 1 / (1 + np.exp(-(intercept + coefficient * X_continuous)))

plt.figure(figsize=(8, 6))
plt.scatter(df['DBO5'], df['Perlidae'], color='blue', alpha=0.6, label='Observed data')
plt.plot(X_continuous, y_curve, color='red', linewidth=2, label='Fitted logistic curve')
plt.title('Logistic Regression Model - Plecoptera: Perlidae', fontsize=14)
plt.xlabel('BOD₅ (mg/l)', fontsize=12)
plt.ylabel('Predicted probability', fontsize=12)
plt.legend(fontsize=12)
equation = f'$P(BOD_5) = \\frac{{1}}{{1 + e^{{-({intercept:.4f} + {coefficient:.4f} \\cdot BOD_5)}}}}$'
plt.text(0.9, 0.8, equation, fontsize=18, color='black')
plt.axvline(x=0, color='black', linestyle='--')
plt.grid(True)
plt.savefig("../../figures/logistic_Perlidae.png", dpi=300, bbox_inches="tight")
plt.show()